In [0]:
SELECT * FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1
WHERE PRODUCT_NAME='Onion'

PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Oil Seeds,Onion,147,2026-02-23T20:40:40.972Z,2026-02-23T21:37:15.136Z


In [0]:
show create table pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1

createtab_stmt
"CREATE TABLE pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1 ( PRODUCTGROUP_NAME STRING, PRODUCT_NAME STRING, PRODUCT_ID BIGINT, lakehouse_inserted_date TIMESTAMP, lakehouse_updated_date TIMESTAMP) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.enableRowTracking' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.domainMetadata' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.feature.rowTracking' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


In [0]:
CREATE TABLE if not EXISTS pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE2 (
  PRODUCTGROUP_NAME STRING,
  PRODUCT_NAME STRING,
  PRODUCT_ID BIGINT,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  lakehouse_inserted_date TIMESTAMP,
  lakehouse_updated_date TIMESTAMP)
USING delta


In [0]:
UPDATE pricing_analytics.silver.daily_pricing_silver
SET PRODUCTGROUP_NAME='Ground Vegetables',
lakehouse_update_date = current_timestamp()
WHERE PRODUCT_NAME='Onion'

num_affected_rows
547


In [0]:
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM pricing_analytics.silver.daily_pricing_silver
WHERE PRODUCT_NAME='Onion'

PRODUCT_NAME,PRODUCTGROUP_NAME
Onion,Ground Vegetables


In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_1 AS
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM pricing_analytics.silver.daily_pricing_silver
WHERE lakehouse_update_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoadScdType2' AND process_status = 'Completed' );

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM pricing_analytics.silver.reporting_dim_product_stage_1

In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_2 AS 
SELECT 
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
  ,goldDim.PRODUCT_NAME AS GOLD_PRODUCT_NAME
  ,goldDim.PRODUCT_ID as GOLD_PRODUCT_ID
 ,ROW_NUMBER() OVER (  ORDER BY silverDim.PRODUCT_NAME,silverDim.PRODUCTGROUP_NAME) as PRODUCT_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM pricing_analytics.silver.reporting_dim_product_stage_1 silverDim
LEFT OUTER JOIN pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE2 goldDim
ON silverDim.PRODUCT_NAME= goldDim.PRODUCT_NAME
and goldDim.end_date IS NULL
WHERE goldDim.PRODUCT_NAME IS NULL OR silverDim.PRODUCTGROUP_NAME <> goldDim.PRODUCTGROUP_NAME

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM pricing_analytics.silver.reporting_dim_product_stage_2

PRODUCT_NAME,PRODUCTGROUP_NAME,GOLD_PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Onion,Ground Vegetables,Onion,1,2026-02-24T15:01:45.942Z,2026-02-24T15:01:45.942Z


In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_3 AS 
SELECT
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
  ,GOLD_PRODUCT_ID
,silverDim.PRODUCT_ID + PREV_MAX_SK_ID as PRODUCT_ID
,PREV_MAX_SK_ID
,case when GOLD_PRODUCT_NAME IS NULL THEN 'New' ELSE 'Changed' END as RECORD_STATUS
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
pricing_analytics.silver.reporting_dim_product_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(PRODUCT_ID),0) as PREV_MAX_SK_ID FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE2 ) goldDim;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM pricing_analytics.silver.reporting_dim_product_stage_3

PRODUCT_NAME,PRODUCTGROUP_NAME,PRODUCT_ID,PREV_MAX_SK_ID,RECORD_STATUS,lakehouse_inserted_date,lakehouse_updated_date
Onion,Ground Vegetables,217,216,Changed,2026-02-24T15:03:07.587Z,2026-02-24T15:03:07.587Z


In [0]:
MERGE INTO pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE2 goldDim
USING pricing_analytics.silver.reporting_dim_product_stage_3 silverDim
ON goldDim.PRODUCT_ID = silverDim.PRODUCT_ID
WHEN MATCHED THEN 
UPDATE SET goldDim.end_date=current_timestamp()
           ,goldDim.lakehouse_updated_date=current_timestamp()
WHEN NOT MATCHED THEN
INSERT (PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,start_date,end_date,lakehouse_inserted_date,lakehouse_updated_date)
VALUES (silverDim.PRODUCTGROUP_NAME,silverDim.PRODUCT_NAME,silverDim.PRODUCT_ID,current_timestamp(),NULL,current_timestamp(),current_timestamp())


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
insert into pricing_analytics.gold.reporting_dim_product_gold_scdtype2
select
PRODUCTGROUP_NAME
,PRODUCT_NAME
,PRODUCT_ID
,current_timestamp()
,NULL 
,current_timestamp() 
,current_timestamp()
from pricing_analytics.silver.reporting_dim_product_stage_3
where RECORD_STATUS = 'Changed'

num_affected_rows,num_inserted_rows
1,1


In [0]:
SELECT * FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE2
WHERE PRODUCT_NAME='Onion'

PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,start_date,end_date,lakehouse_inserted_date,lakehouse_updated_date
Oil Seeds,Onion,147,2026-02-24T14:23:43.487Z,2026-02-24T14:44:54.444Z,2026-02-24T14:23:43.487Z,2026-02-24T14:44:54.444Z
Ground Vegetables,Onion,217,2026-02-24T15:40:24.953Z,null,2026-02-24T15:40:24.953Z,2026-02-24T15:40:24.953Z
Ground Vegetables,Onion,217,2026-02-24T15:40:01.685Z,null,2026-02-24T15:40:01.685Z,2026-02-24T15:40:01.685Z
Vegetables,Onion,216,2026-02-24T14:55:34.043Z,null,2026-02-24T14:55:34.043Z,2026-02-24T14:55:34.043Z


In [0]:
INSERT INTO  pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESSED_TABLE_DATETIME,PROCESS_STATUS)
SELECT 'reportingDimensionTablesLoadScdType2' , max(lakehouse_update_date) ,'Completed' FROM pricing_analytics.silver.daily_pricing_silver

num_affected_rows,num_inserted_rows
1,1
